# 03 · Classify documents in a storage account

Classify documents that live in Azure Blob Storage. The trained classifier reads
the input document by URL, and URL input can't use the resource's managed identity,
so we mint a **short-lived user-delegation SAS** per blob (no account key).

`split='none'` treats each file as a single document and returns one class for it.
Use `split='auto'` for files that concatenate several documents.

## 1. Setup

In [ ]:
import os, sys
from pathlib import Path

_here = Path.cwd()
for _c in (_here, _here / 'notebooks', *_here.parents):
    if (_c / '_lib.py').exists():
        sys.path.insert(0, str(_c))
        break
import _lib
from azure.ai.documentintelligence.models import ClassifyDocumentRequest

di = _lib.get_di_client()
classifier_id = _lib.CLASSIFIER_ID
ACCOUNT = os.environ.get('STORAGE_ACCOUNT_NAME', '')

## 2. Classify a single blob

Any blob in the training container works. The classifier returns the `docType`
and a **confidence** score you can threshold on.

In [ ]:
blob_name = 'invoice/FabrikamInvoice_1.pdf'
sas_url = _lib.blob_sas_url(blob_name, account_name=ACCOUNT)

poller = di.begin_classify_document(
    classifier_id,
    ClassifyDocumentRequest(url_source=sas_url),
    split='none',
)
result = poller.result()
_lib.print_result(result, source=blob_name)
_lib.save_result(result, _lib.SAMPLE_OUTPUT_DIR / 'storage_invoice.json')

## 3. Classify one document per class

Includes the handwritten claim to show it still routes to `claim`.

In [ ]:
blobs = [
    'invoice/FabrikamInvoice_1.pdf',
    'claim/ClaimForm_1.pdf',
    'claim/claimform_handwritten_1.pdf',
]

for blob_name in blobs:
    sas_url = _lib.blob_sas_url(blob_name, account_name=ACCOUNT)
    result = di.begin_classify_document(
        classifier_id,
        ClassifyDocumentRequest(url_source=sas_url),
        split='none',
    ).result()
    _lib.print_result(result, source=blob_name)
    print()

Cached responses land in [`reference/sample-output/`](../reference/sample-output).
To classify files straight from disk instead, see
[`04_classify_local_file.ipynb`](04_classify_local_file.ipynb).